# Моделирование: прогноз суточного PM2.5

**Вопрос.** Насколько точно можно предсказать среднесуточную концентрацию PM2.5 по вчерашнему значению, погоде и календарю, и лучше ли модели простого правила «завтра как сегодня»?

**Цель:** `pm25_mean` (µg/m³) из `data/processed/daily.csv`.

**Модели (L10–L11):**
- базовые ориентиры: **наивный прогноз** («завтра как сегодня» — `pm25_lag1`) и **среднее по месяцу** обучающей выборки (`month_mean`, ориентир для моделей без лага);
- **Linear Regression** (`StandardScaler` + `LinearRegression` в `Pipeline`) и **Random Forest** (`RandomForestRegressor`) — каждая в двух вариантах: со вчерашним `pm25_lag1` и без него («только погода и календарь»).

**Почему разбиение по времени, а не `train_test_split(shuffle=True)`:** соседние сутки сильно связаны по погоде и загрязнению. Случайное перемешивание перемешало бы прошлое и будущее: в обучении оказались бы дни рядом с тестовыми, что дало бы подглядывание в будущее и заниженную ошибку.

**Почему тест трогаем один раз:** если подбирать признаки, дату разбиения или гиперпараметры по метрикам теста, тестовая часть перестаёт быть независимой проверкой. Все решения принимаются только на `train` (через скользящую проверку по времени), а на `test` модели оцениваются один раз.

**Метрики:**
- `MAE` = среднее `|факт − прогноз|` (µg/m³, основная метрика);
- `RMSE` = √(среднее `(факт − прогноз)²`) — сильнее штрафует крупные промахи;
- `R²` = `1 − Σ(факт − прогноз)² / Σ(факт − среднее факта)²` — доля разброса, которую объясняет прогноз (может быть `< 0`).

**Оговорка о погоде:** в признаках используется фактическая суточная погода дня `D`, а не метеопрогноз. Поэтому результаты показывают «оценку сверху» для настоящего прогноза на завтра.

In [1]:
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sys.path.append("..")
from src import features, hypotheses, models

DAILY_PATH = "../data/processed/daily.csv"
FIGURES_DIR = "../figures"

# Целевая переменная: среднесуточный PM2.5 (µg/m³)
TARGET = "pm25_mean"

# Полный набор признаков (погода, календарные флаги и вчерашний PM2.5)
FEATURES = [
    "temp_c",
    "wind_speed",
    "humidity",
    "precip",
    "pressure",
    "heating_season",
    "is_weekend",
    "pm25_lag1",
]

# Набор без вчерашнего PM2.5: проверяем вклад только погоды и календаря
FEATURES_NO_LAG = [
    "temp_c",
    "wind_speed",
    "humidity",
    "precip",
    "pressure",
    "heating_season",
    "is_weekend",
]

# Граница разбиения: с 01.07.2024 начинается тестовый период с полным отопительным сезоном 2024/25
TEST_START = "2024-07-01"

# Число фолдов скользящей проверки по времени на train
N_SPLITS = 5

# Фиксируем зерно случайности для леса, bootstrap и перестановочной важности
SEED = 42
N_BOOT = 5000
N_REPEATS = 30

# Единая тема графиков: палитра, удобная для дальтоников
sns.set_theme(style="whitegrid", context="talk", palette="colorblind")

## Шаг 1. Таблица для модели

Обе модели (Linear Regression и Random Forest) учатся на одном наборе признаков, чтобы их сравнение было честным: метеопараметры дня (`temp_c`, `wind_speed`, `humidity`, `precip`, `pressure`), календарные флаги (`heating_season`, `is_weekend`) и вчерашний уровень загрязнения (`pm25_lag1`). Булевы флаги переводятся в `0.0` и `1.0`.

**Почему не берём часть колонок из `daily.csv`:**
- `month` (1…12) и `dayofweek` (0…6) как числа бессмысленны для линейной модели: декабрь (`12`) и январь (`1`) одинаково грязные, но стоят на разных концах шкалы, а коэффициент «на +1 месяц» нельзя объяснить. Сезонность уже несут `heating_season` и `temp_c`, а разницу будних и выходных дней — `is_weekend`.
- `is_calm` — это жёсткий порог от `wind_speed` (`< 1 м/с`), то есть дубль скорости ветра.
- `season` (текст) потребовал бы one-hot кодирования, тогда как сезонность уже учтена.

**Почему удаляем строки с пропусками, а не заполняем:** модели должны учиться и проверяться только на реальных измеренных сутках без выдуманных значений. Все шесть прогнозов (включая модели без лага) считаются на одних и тех же строках, чтобы сравнивать их на одинаковых сутках.

**Почему `pm25_lag1` не даёт утечки данных:** по построению в `features.lag_previous_day` это среднее PM2.5 за предыдущие календарные сутки (`D−1`), известное до начала дня `D`.

In [2]:
daily = features.read_daily_csv(DAILY_PATH)
features.check_daily(daily)

n_total = len(daily)
n_no_target = int(daily[TARGET].isna().sum())
n_no_lag_only = int((daily[TARGET].notna() & daily["pm25_lag1"].isna()).sum())

table = models.make_model_table(daily, FEATURES, TARGET)
assert len(table) == 1414, f"ожидалось 1414 строк, получено {len(table)}"

print(f"Всего суток в daily.csv: {n_total}")
print(f"Без цели (pm25_mean): {n_no_target}")
print(f"С целью, но без вчерашнего значения (pm25_lag1): {n_no_lag_only}")
print(f"Всего потеряно строк: {n_total - len(table)}")
print(f"Строк в таблице модели: {len(table)}")

Всего суток в daily.csv: 2044
Без цели (pm25_mean): 510
С целью, но без вчерашнего значения (pm25_lag1): 120
Всего потеряно строк: 630
Строк в таблице модели: 1414


## Шаг 2. Разбиение по времени

Разбиваем таблицу по границе `TEST_START = "2024-07-01"`: все сутки строго до неё идут в обучение (`train`), начиная с неё — в тест (`test`).

**Почему `2024-07-01`, а не `2025-07-01`:** данные станции заканчиваются `2025-11-13` (после `2025-11-14` станция не передаёт измерения). При границе `2025-07-01` в тесте остались бы только лето, осень и начало отопительного сезона без зимы — тогда модели главной проблемы (зимнего смога) не проверялись бы. Граница `2024-07-01` оставляет в `test` полный отопительный сезон `2024/25` вместе с летом и осенью `2025` года, а в `train` — данные `2020–2024` годов.

Решение о границе разбиения принято **до** расчёта каких-либо метрик на тесте.

In [3]:
train, test = models.split_by_date(table, TEST_START)

test_share = len(test) / len(table)
train_heating = int((train["heating_season"] == 1.0).sum())
test_heating = int((test["heating_season"] == 1.0).sum())

# Считаем, сколько календарных суток в тестовом периоде не попало в таблицу модели
test_calendar = daily.loc[daily["date"] >= pd.Timestamp(TEST_START), "date"]
test_missing_days = len(test_calendar) - len(test)

print(
    f"train: n = {len(train)} ({1 - test_share:.1%}), "
    f"{train['date'].min():%Y-%m-%d} ... {train['date'].max():%Y-%m-%d}, "
    f"отопительных суток: {train_heating}"
)
print(
    f"test:  n = {len(test)} ({test_share:.1%}), "
    f"{test['date'].min():%Y-%m-%d} ... {test['date'].max():%Y-%m-%d}, "
    f"отопительных суток: {test_heating}"
)
print(
    f"Календарных суток в периоде теста: {len(test_calendar)}, "
    f"пропущено в таблице модели: {test_missing_days}"
)

train: n = 931 (65.8%), 2020-04-11 ... 2024-06-30, отопительных суток: 504
test:  n = 483 (34.2%), 2024-07-01 ... 2025-11-13, отопительных суток: 213
Календарных суток в периоде теста: 501, пропущено в таблице модели: 18


**Что видим.** Из `2 044` суток в `daily.csv` потеряно `630` строк: `510` суток без цели `pm25_mean` и ещё `120` суток с целью, но без вчерашнего значения `pm25_lag1` (вывод шага 1). В итоговой таблице модели остаётся `1 414` суток с полным набором признаков и целью. После разбиения по границе `2024-07-01` обучающая часть `train` содержит `931` сутки (`65.8%`, `2020-04-11 … 2024-06-30`, из них `504` отопительных суток), а тестовая часть `test` — `483` суток (`34.2%`, `2024-07-01 … 2025-11-13`, из них `213` отопительных суток; вывод шага 2). Из `501` календарных суток тестового периода в таблицу модели не попали `18` суток из-за пропусков цели или лага (вывод шага 2).

## Шаг 3. Скользящая проверка по времени на train

Один контрольный отрезок даёт шумную оценку ошибки, а выбирать параметры модели по отложенной контрольной выборке нельзя — иначе итоговая проверка перестанет быть независимой. Поэтому качество и настройки моделей оцениваем только внутри `train` с помощью скользящей проверки по времени — `TimeSeriesSplit(n_splits=5)` ⚠️ (расширяющееся окно).

**Как устроен `TimeSeriesSplit`:** делим `train` по порядку времени на 6 последовательных кусков:
- фолд 1 — учимся на 1-м куске, проверяем на 2-м;
- фолд 2 — учимся на 1–2-м, проверяем на 3-м;
- и так далее до 5-го фолда — модель всегда учится на прошлом и проверяется на следующем куске.

Разбиение идёт по числу строк, поэтому из-за пропуска данных в 2022 году календарная длина кусков различается — ниже печатаем точные даты границ. В каждом фолде `mae_naive` — это ошибка наивного прогноза («завтра как сегодня», `pm25_lag1`) на том же проверочном куске.

In [4]:
# Проверяем линейную регрессию с лагом и без лага на фолдах train
cv_linreg = models.time_cv_table(train, FEATURES, models.make_linear_model(), N_SPLITS)
cv_linreg_no_lag = models.time_cv_table(
    train, FEATURES_NO_LAG, models.make_linear_model(), N_SPLITS
)

print("Linear Regression (с pm25_lag1) — скользящая проверка на train:")
print(cv_linreg.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

print("\nLinear Regression (без pm25_lag1) — скользящая проверка на train:")
print(cv_linreg_no_lag.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

Linear Regression (с pm25_lag1) — скользящая проверка на train:
 fold  train_end  val_start    val_end  n_train  n_val  mae_model  mae_naive
    1 2021-01-26 2021-02-04 2022-02-05      156    155     13.869     14.471
    2 2022-02-05 2022-02-06 2022-12-28      311    155     12.043     13.321
    3 2022-12-28 2022-12-29 2023-08-11      466    155      9.959      8.240
    4 2023-08-11 2023-08-12 2024-01-25      621    155     11.637     12.486
    5 2024-01-25 2024-01-26 2024-06-30      776    155      9.061      9.291

Linear Regression (без pm25_lag1) — скользящая проверка на train:
 fold  train_end  val_start    val_end  n_train  n_val  mae_model  mae_naive
    1 2021-01-26 2021-02-04 2022-02-05      156    155     24.306     14.471
    2 2022-02-05 2022-02-06 2022-12-28      311    155     16.750     13.321
    3 2022-12-28 2022-12-29 2023-08-11      466    155     15.717      8.240
    4 2023-08-11 2023-08-12 2024-01-25      621    155     14.597     12.486
    5 2024-01-25 2024-

## Шаг 4. Подбор Random Forest только на train

Для `RandomForestRegressor` подбираем три гиперпараметра на расширяющемся окне внутри `train`:
- `n_estimators` — число деревьев в лесу;
- `max_depth` — максимальная глубина каждого дерева;
- `min_samples_leaf` — минимальное число суток в листе дерева.

Перебираем сетку из 18 комбинаций (`{100, 300} × {4, 8, 16} × {1, 5, 20}`) отдельно для версии с лагом и без него. Лучшей считаем комбинацию с минимальным средним MAE по 5 фолдам (`cv_mae_mean`). Значение `None` для `max_depth` не используем: в `DataFrame` колонка с `None` и числами становится `float` с `NaN`, и лучшую строку нельзя передать обратно в модель как целые числа.

In [5]:
# Перебираем сетку гиперпараметров Random Forest только на обучающей выборке
FOREST_GRID = models.make_forest_grid([100, 300], [4, 8, 16], [1, 5, 20])

grid_lag = models.forest_grid_cv(train, FEATURES, FOREST_GRID, N_SPLITS, SEED)
grid_no_lag = models.forest_grid_cv(train, FEATURES_NO_LAG, FOREST_GRID, N_SPLITS, SEED)

print("Топ-5 сетки Random Forest (с pm25_lag1):")
print(grid_lag.head(5).to_string(index=False, float_format=lambda x: f"{x:.3f}"))

print("\nТоп-5 сетки Random Forest (без pm25_lag1):")
print(grid_no_lag.head(5).to_string(index=False, float_format=lambda x: f"{x:.3f}"))

# Извлекаем лучшие параметры как обычные int из первой строки отсортированной таблицы
best_lag = {
    "n_estimators": int(grid_lag.loc[0, "n_estimators"]),
    "max_depth": int(grid_lag.loc[0, "max_depth"]),
    "min_samples_leaf": int(grid_lag.loc[0, "min_samples_leaf"]),
}
best_no_lag = {
    "n_estimators": int(grid_no_lag.loc[0, "n_estimators"]),
    "max_depth": int(grid_no_lag.loc[0, "max_depth"]),
    "min_samples_leaf": int(grid_no_lag.loc[0, "min_samples_leaf"]),
}

print("\nВыбранные параметры (с лагом):", best_lag)
print("Выбранные параметры (без лага):", best_no_lag)

# Считаем ошибки по фолдам для лучших лесов
cv_forest = models.time_cv_table(
    train, FEATURES, models.make_forest(**best_lag, seed=SEED), N_SPLITS
)
cv_forest_no_lag = models.time_cv_table(
    train, FEATURES_NO_LAG, models.make_forest(**best_no_lag, seed=SEED), N_SPLITS
)

print("\nRandom Forest (с pm25_lag1) — фолды с лучшими параметрами:")
print(cv_forest.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

print("\nRandom Forest (без pm25_lag1) — фолды с лучшими параметрами:")
print(cv_forest_no_lag.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

# Сводим средний, минимальный и максимальный MAE по фолдам для всех вариантов
cv_sources = [
    ("naive", cv_linreg, "mae_naive"),
    ("linreg", cv_linreg, "mae_model"),
    ("forest", cv_forest, "mae_model"),
    ("linreg_no_lag", cv_linreg_no_lag, "mae_model"),
    ("forest_no_lag", cv_forest_no_lag, "mae_model"),
]
summary_rows = []
for name, frame, col in cv_sources:
    vals = frame[col]
    better_folds = int((frame[col] < frame["mae_naive"]).sum())
    summary_rows.append(
        {
            "model": name,
            "cv_mae_mean": float(vals.mean()),
            "cv_mae_min": float(vals.min()),
            "cv_mae_max": float(vals.max()),
            "folds_better_than_naive": better_folds,
        }
    )

cv_summary = pd.DataFrame(summary_rows)
print("\nСводка скользящей проверки на train (5 фолдов):")
print(cv_summary.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

Топ-5 сетки Random Forest (с pm25_lag1):
 n_estimators  max_depth  min_samples_leaf  cv_mae_mean  cv_mae_std
          300          8                 5        9.666       2.675
          300         16                 5        9.673       2.662
          300          8                 1        9.674       2.531
          300         16                 1        9.703       2.451
          300          4                 5        9.756       2.649

Топ-5 сетки Random Forest (без pm25_lag1):
 n_estimators  max_depth  min_samples_leaf  cv_mae_mean  cv_mae_std
          300          8                 5       12.326       3.719
          100          8                 5       12.328       3.730
          100         16                 5       12.332       3.720
          300         16                 5       12.336       3.705
          300          4                 5       12.356       3.706

Выбранные параметры (с лагом): {'n_estimators': 300, 'max_depth': 8, 'min_samples_leaf': 5}
Выбран


Random Forest (с pm25_lag1) — фолды с лучшими параметрами:
 fold  train_end  val_start    val_end  n_train  n_val  mae_model  mae_naive
    1 2021-01-26 2021-02-04 2022-02-05      156    155     12.952     14.471
    2 2022-02-05 2022-02-06 2022-12-28      311    155     11.170     13.321
    3 2022-12-28 2022-12-29 2023-08-11      466    155      6.506      8.240
    4 2023-08-11 2023-08-12 2024-01-25      621    155     10.312     12.486
    5 2024-01-25 2024-01-26 2024-06-30      776    155      7.392      9.291

Random Forest (без pm25_lag1) — фолды с лучшими параметрами:
 fold  train_end  val_start    val_end  n_train  n_val  mae_model  mae_naive
    1 2021-01-26 2021-02-04 2022-02-05      156    155     17.002     14.471
    2 2022-02-05 2022-02-06 2022-12-28      311    155     15.377     13.321
    3 2022-12-28 2022-12-29 2023-08-11      466    155      8.366      8.240
    4 2023-08-11 2023-08-12 2024-01-25      621    155     11.268     12.486
    5 2024-01-25 2024-01-26 202

**Что видим.** (вывод шагов 3–4)

- **Разброс по фолдам показывает «шум» одной проверки:** на 5 фолдах внутри `train` (по `155` суток проверки в каждом) ошибка сильно меняется от периода к периоду: у наивного прогноза `MAE` колеблется от `8.240` (фолд 3) до `14.471` µg/m³ (фолд 1), у `linreg` — от `9.061` до `13.869`, у `forest` — от `6.506` до `12.952`, у `linreg_no_lag` — от `12.837` до `24.306`, у `forest_no_lag` — от `8.366` до `17.002` µg/m³.
- **Выбранные параметры Random Forest:** для обеих версий (с `pm25_lag1` и без `pm25_lag1`) минимальный средний `MAE` на `train` дала одна и та же комбинация `n_estimators = 300`, `max_depth = 8`, `min_samples_leaf = 5` (`cv_mae_mean = 9.666` и `12.326` µg/m³ соответственно).
- **Сравнение с наивным прогнозом в CV:**
  - `forest` (среднее `9.666` против `11.562` µg/m³ у `naive`) точнее наивного во всех `5` из `5` фолдов;
  - `linreg` (среднее `11.314` µg/m³) точнее наивного в `4` из `5` фолдов (уступает на фолде 3: `9.959` против `8.240` µg/m³);
  - без вчерашнего `pm25_lag1` ошибка выше: `forest_no_lag` (среднее `12.326` µg/m³) точнее наивного только в `1` из `5` фолдов (фолд 4: `11.268` против `12.486` µg/m³), а `linreg_no_lag` (среднее `16.841` µg/m³) — ни в одном (`0` из `5` фолдов).

## Шаг 5. Оценка на тесте (один раз)

Переобучаем все четыре модели на всей обучающей выборке `train` (`931` сутки) с параметрами, выбранными в шаге 4 (`n_estimators = 300`, `max_depth = 8`, `min_samples_leaf = 5`), и вычисляем прогнозы на отложенной тестовой части `test` (`483` суток).

Это **единственная ячейка**, где считаются прогнозы на тесте: все дальнейшие сравнения, графики и разбор ошибок на шагах 6–9 только читают сохранённую таблицу `test_predictions` и не переобучают модели.

In [6]:
# Порядок всех шести прогнозов в сводных таблицах
MODEL_NAMES = ["naive", "month_mean", "linreg", "forest", "linreg_no_lag", "forest_no_lag"]

# Признаки каждой из четырёх обучаемых моделей
MODEL_FEATURES = {
    "linreg": FEATURES,
    "forest": FEATURES,
    "linreg_no_lag": FEATURES_NO_LAG,
    "forest_no_lag": FEATURES_NO_LAG,
}

# Обучаем все четыре модели на полном train один раз
unfitted_models = {
    "linreg": models.make_linear_model(),
    "forest": models.make_forest(**best_lag, seed=SEED),
    "linreg_no_lag": models.make_linear_model(),
    "forest_no_lag": models.make_forest(**best_no_lag, seed=SEED),
}
final_models = {}
for name, model in unfitted_models.items():
    cols = MODEL_FEATURES[name]
    final_models[name] = model.fit(train[cols], train[TARGET])

# Собираем прогнозы на тесте в одну таблицу (больше нигде не предсказываем на test)
test_predictions = test[["date", TARGET, "heating_season"]].copy()
test_predictions["naive"] = models.naive_forecast(test)
test_predictions["month_mean"] = models.month_mean_forecast(train, test, TARGET)
for name, model in final_models.items():
    test_predictions[name] = model.predict(test[MODEL_FEATURES[name]])

assert not test_predictions[MODEL_NAMES].isna().any().any(), "в прогнозах на тесте есть NaN"

# Считаем прогнозы на train, чтобы видеть разницу ошибок train/test (признак переобучения)
train_predictions = {
    "naive": models.naive_forecast(train),
    "month_mean": models.month_mean_forecast(train, train, TARGET),
}
for name, model in final_models.items():
    train_predictions[name] = model.predict(train[MODEL_FEATURES[name]])

# Сводная таблица метрик на тесте + MAE на train
metric_rows = []
for name in MODEL_NAMES:
    m_test = models.regression_metrics(test_predictions[TARGET], test_predictions[name])
    m_train = models.regression_metrics(train[TARGET], train_predictions[name])
    metric_rows.append(
        {
            "model": name,
            "n": m_test["n"],
            "mae": m_test["mae"],
            "rmse": m_test["rmse"],
            "r2": m_test["r2"],
            "mae_train": m_train["mae"],
        }
    )

test_metrics = pd.DataFrame(metric_rows)
assert len(test_metrics) == 6, "в таблице метрик должно быть 6 строк"
assert (test_metrics["n"] == len(test)).all(), "число суток должно совпадать со всеми строками test"

print("Метрики шести прогнозов на тесте (и MAE на train):")
print(test_metrics.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

# Отдельно считаем MAE в отопительный сезон (heating_season == 1) и вне его (== 0)
season_rows = []
for name in MODEL_NAMES:
    for season_flag in [1, 0]:
        sub = test_predictions[test_predictions["heating_season"] == float(season_flag)]
        m_sub = models.regression_metrics(sub[TARGET], sub[name])
        season_rows.append(
            {
                "model": name,
                "heating_season": season_flag,
                "n": m_sub["n"],
                "mae": m_sub["mae"],
            }
        )

season_mae = pd.DataFrame(season_rows)
assert len(season_mae) == 12, "в таблице по сезонам должно быть 12 строк (6 моделей * 2 сезона)"
assert season_mae.groupby("model")["n"].sum().eq(len(test)).all(), "сумма суток по сезонам != len(test)"

print("\nMAE на тесте по сезонам (heating_season: 1 = отопительный, 0 = вне сезона):")
print(season_mae.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

Метрики шести прогнозов на тесте (и MAE на train):
        model   n    mae   rmse    r2  mae_train
        naive 483  8.592 14.828 0.581     11.906
   month_mean 483  9.969 16.921 0.454     13.335
       linreg 483  8.921 13.054 0.675     10.906
       forest 483  8.097 13.653 0.644      6.552
linreg_no_lag 483 12.830 16.939 0.453     14.748
forest_no_lag 483 10.765 18.397 0.354      8.201

MAE на тесте по сезонам (heating_season: 1 = отопительный, 0 = вне сезона):
        model  heating_season   n    mae
        naive               1 213 15.419
        naive               0 270  3.206
   month_mean               1 213 18.427
   month_mean               0 270  3.296
       linreg               1 213 13.725
       linreg               0 270  5.132
       forest               1 213 14.583
       forest               0 270  2.981
linreg_no_lag               1 213 17.841
linreg_no_lag               0 270  8.877
forest_no_lag               1 213 20.125
forest_no_lag               0 270  3.

**Что видим.** (вывод шага 5)

- **Метрики всех шести прогнозов на тесте (`n = 483`):**
  - `naive`: `MAE = 8.592` µg/m³, `RMSE = 14.828` µg/m³, `R² = 0.581`;
  - `month_mean`: `MAE = 9.969` µg/m³, `RMSE = 16.921` µg/m³, `R² = 0.454`;
  - `linreg`: `MAE = 8.921` µg/m³, `RMSE = 13.054` µg/m³, `R² = 0.675`;
  - `forest`: `MAE = 8.097` µg/m³, `RMSE = 13.653` µg/m³, `R² = 0.644`;
  - `linreg_no_lag`: `MAE = 12.830` µg/m³, `RMSE = 16.939` µg/m³, `R² = 0.453`;
  - `forest_no_lag`: `MAE = 10.765` µg/m³, `RMSE = 18.397` µg/m³, `R² = 0.354`.
  Окончательное сравнение моделей с наивным прогнозом и проверка значимости различий выполняются в шаге 6.
- **Разница между `train` и `test` у Random Forest (признак переобучения):** на `train` ошибка `forest` составляет `6.552` против `8.097` µg/m³ на `test`, у `forest_no_lag` — `8.201` против `10.765` µg/m³: деревья подстраиваются под обучающую выборку сильнее линейных моделей, у которых `mae_train` на этом наборе даже выше тестового (`10.906` против `8.921` у `linreg` и `14.748` против `12.830` у `linreg_no_lag`).
- **Ошибки по сезонам:** во всех шести прогнозах `MAE` в отопительный сезон (`n = 213`) кратно выше, чем вне его (`n = 270`): `naive` — `15.419` против `3.206`, `month_mean` — `18.427` против `3.296`, `linreg` — `13.725` против `5.132`, `forest` — `14.583` против `2.981`, `linreg_no_lag` — `17.841` против `8.877`, `forest_no_lag` — `20.125` против `3.381` µg/m³.
- **Отказ от лага `pm25_lag1`:** при использовании только погоды и календарных флагов у обеих моделей `MAE` на тесте оказывается выше, чем у среднего по месяцу `month_mean` (`12.830` у `linreg_no_lag` и `10.765` у `forest_no_lag` против `9.969` µg/m³), а `R²` не превышает `month_mean` (`0.453` и `0.354` против `0.454`).

## Шаг 6. Точнее ли модель наивного прогноза?

Чтобы проверить, отличается ли ошибка модели на тесте от базовых ориентиров (наивного прогноза `naive` и среднего по месяцу `month_mean`), применяем **парный bootstrap по суткам** (L8): `N_BOOT = 5000` раз случайно выбираем с возвращением **одни и те же** индексы суток для обеих моделей и считаем `MAE_model − MAE_reference`.

**Как читать 95 % доверительный интервал (перцентильный CI):**
- весь интервал строго меньше нуля (`ci_high < 0`) — модель точнее ориентира на этом тесте;
- интервал включает ноль — различие в средней ошибке на этих данных не доказано;
- `share_below_zero` — доля bootstrap-выборок, где `MAE_model < MAE_reference`.

**Оговорка:** соседние сутки во временном ряду зависимы, тогда как bootstrap перевыбирает отдельные сутки независимо. Поэтому доверительные интервалы получаются несколько узковатыми; формальные p-value и поправки на множественность здесь не считаем.

In [7]:
# Пары (модель, ориентир): 5 прогнозов против naive + 2 модели без лага против month_mean
COMPARISONS = [
    ("month_mean", "naive"),
    ("linreg", "naive"),
    ("forest", "naive"),
    ("linreg_no_lag", "naive"),
    ("forest_no_lag", "naive"),
    ("linreg_no_lag", "month_mean"),
    ("forest_no_lag", "month_mean"),
]

mae_by_model = test_metrics.set_index("model")["mae"]
diff_rows = []
for model_name, ref_name in COMPARISONS:
    diffs = models.bootstrap_mae_diff(
        test_predictions[TARGET],
        test_predictions[model_name],
        test_predictions[ref_name],
        N_BOOT,
        SEED,
    )
    ci_low, ci_high = hypotheses.percentile_ci(diffs)
    mae_diff = float(mae_by_model[model_name] - mae_by_model[ref_name])
    # Проверяем совпадение разности MAE с таблицей test_metrics
    exact_diff = float(
        np.abs(test_predictions[TARGET] - test_predictions[model_name]).mean()
        - np.abs(test_predictions[TARGET] - test_predictions[ref_name]).mean()
    )
    assert abs(mae_diff - exact_diff) < 1e-9, "mae_diff не совпадает с test_metrics"
    assert ci_low <= ci_high, "нижняя граница CI должна быть <= верхней"
    diff_rows.append(
        {
            "model": model_name,
            "reference": ref_name,
            "mae_diff": mae_diff,
            "ci_low": ci_low,
            "ci_high": ci_high,
            "share_below_zero": float((diffs < 0).mean()),
        }
    )

mae_diff_table = pd.DataFrame(diff_rows)
assert len(mae_diff_table) == 7, "в таблице сравнений должно быть 7 строк"

print("Парный bootstrap 95 % CI разности MAE (model - reference, µg/m³):")
print(mae_diff_table.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

Парный bootstrap 95 % CI разности MAE (model - reference, µg/m³):
        model  reference  mae_diff  ci_low  ci_high  share_below_zero
   month_mean      naive     1.377   0.259    2.508             0.009
       linreg      naive     0.330  -0.270    0.910             0.140
       forest      naive    -0.495  -1.326    0.313             0.887
linreg_no_lag      naive     4.238   3.035    5.364             0.000
forest_no_lag      naive     2.173   0.887    3.469             0.000
linreg_no_lag month_mean     2.861   1.944    3.742             0.000
forest_no_lag month_mean     0.796  -0.454    2.068             0.108


## Шаг 7. Факт и прогноз на тесте (график 9)

Сравниваем по датам тестового периода фактический суточный `pm25_mean` с прогнозами `Linear Regression` и `Random Forest` (варианты с лагом `pm25_lag1`) и ориентиром суточной нормы ВОЗ (`15 µg/m³`). Наивный прогноз на график не выводим: это фактический ряд, сдвинутый на одни сутки, поэтому линии почти сливаются; его `MAE` указан в заголовке.

In [8]:
mae_rf = float(mae_by_model["forest"])
mae_lr = float(mae_by_model["linreg"])
mae_naive = float(mae_by_model["naive"])

palette = sns.color_palette()
fig, (ax_lr, ax_rf) = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

ax_lr.plot(
    test_predictions["date"],
    test_predictions[TARGET],
    color="black",
    linewidth=1.5,
    alpha=0.85,
    label="Факт",
)
ax_lr.plot(
    test_predictions["date"],
    test_predictions["linreg"],
    color=palette[0],
    linewidth=1.5,
    alpha=0.85,
    label="Linear Regression",
)
ax_lr.axhline(
    features.WHO_DAILY_LIMIT,
    color="gray",
    linestyle="--",
    linewidth=1.2,
    label=f"Норма ВОЗ ({features.WHO_DAILY_LIMIT} µg/m³)",
)
ax_lr.set_ylabel("PM2.5, µg/m³")
ax_lr.set_title("Linear Regression (с лагом pm25_lag1)")
ax_lr.legend(loc="upper right")

ax_rf.plot(
    test_predictions["date"],
    test_predictions[TARGET],
    color="black",
    linewidth=1.5,
    alpha=0.85,
    label="Факт",
)
ax_rf.plot(
    test_predictions["date"],
    test_predictions["forest"],
    color=palette[1],
    linewidth=1.5,
    alpha=0.85,
    label="Random Forest",
)
ax_rf.axhline(
    features.WHO_DAILY_LIMIT,
    color="gray",
    linestyle="--",
    linewidth=1.2,
    label=f"Норма ВОЗ ({features.WHO_DAILY_LIMIT} µg/m³)",
)
ax_rf.set_xlabel("Дата")
ax_rf.set_ylabel("PM2.5, µg/m³")
ax_rf.set_title("Random Forest (с лагом pm25_lag1)")
ax_rf.legend(loc="upper right")

fig.suptitle(
    f"На тесте Random Forest ошибается в среднем на {mae_rf:.1f} µg/m³, "
    f"Linear Regression — на {mae_lr:.1f}, наивный прогноз — на {mae_naive:.1f}",
    y=1.01,
)
fig.tight_layout()
fig.savefig(f"{FIGURES_DIR}/09_forecast.png", dpi=150, bbox_inches="tight")
plt.close(fig)

**Что видим.** (вывод шагов 6–7)

- **Сравнение с наивным прогнозом и средним по месяцу (парный bootstrap, 95 % CI разности MAE в µg/m³):**
  - `forest` vs `naive`: разность `-0.495`, 95 % CI `[-1.326, 0.313]`, доля `< 0` — `0.887` (интервал содержит ноль: точечно MAE у леса ниже, но статистически уверенное преимущество над наивным прогнозом на этом тесте не доказано);
  - `linreg` vs `naive`: разность `+0.330`, 95 % CI `[-0.270, 0.910]`, доля `< 0` — `0.140` (интервал включает ноль: различие с наивным не доказано);
  - `month_mean` vs `naive`: разность `+1.377`, 95 % CI `[0.259, 2.508]`, доля `< 0` — `0.009` (`month_mean` уступает наивному прогнозу);
  - `linreg_no_lag` vs `naive`: разность `+4.238`, 95 % CI `[3.035, 5.364]`, доля `< 0` — `0.000` (существенно хуже наивного);
  - `forest_no_lag` vs `naive`: разность `+2.173`, 95 % CI `[0.887, 3.469]`, доля `< 0` — `0.000` (существенно хуже наивного);
  - `linreg_no_lag` vs `month_mean`: разность `+2.861`, 95 % CI `[1.944, 3.742]`, доля `< 0` — `0.000` (хуже среднего по месяцу);
  - `forest_no_lag` vs `month_mean`: разность `+0.796`, 95 % CI `[-0.454, 2.068]`, доля `< 0` — `0.108` (интервал включает ноль: модель без лага не превосходит простое среднее по месяцу).
- **График факта и прогноза на тесте:** обе модели с лагом повторяют сезонный подъём выше нормы ВОЗ (`15 µg/m³`) и летний спад. Главные промахи приходятся на зимние пики отопительного сезона (конец декабря 2024 — февраль 2025, что согласуется с шагом 5: `MAE = 13.725` у `linreg` и `14.583` у `forest` в отопительный сезон против `5.132` и `2.981` µg/m³ вне его). На графике видно также, что `Linear Regression` занижает летние значения вплоть до отрицательных прогнозов в отдельные дни, а `Random Forest` остаётся в положительной области, но иногда завышает отдельные зимние пики.

![График 9](../figures/09_forecast.png)

## Шаг 8. На что опираются модели

Разбираем, какие признаки используют линейная регрессия и случайный лес.

**Коэффициенты Linear Regression:**
- `coef_std` — изменение прогноза PM2.5 (µg/m³) при росте признака на 1 стандартное отклонение (`std`); удобно сравнивать признаки в разных единицах между собой;
- `coef_original` — изменение прогноза (µg/m³) при росте признака на 1 исходную единицу (°C, м/с, %, мм, гПа, µg/m³); для бинарных флагов `0/1` (`heating_season`, `is_weekend`) — это разница прогноза при прочих равных условиях.

Стандартные `p`-значения коэффициентов не считаем: остатки модели автокоррелированы по времени (проверим это в шаге 9), поэтому обычные формулы стандартных ошибок неверны.

**Перестановочная важность Random Forest ⚠️:** обученную модель не переобучаем; на тестовой выборке по очереди перемешиваем значения одного столбца (разрушаем его связь с целью) и измеряем, на сколько µg/m³ вырастает `MAE`. Важность считаем именно на `test`, а не на `train`: на обучающих данных лес запоминает выборку и завышает вклад шумных признаков, тогда как на отложенных данных видно, на что модель действительно опирается на новых сутках. Это не подбор модели — все параметры зафиксированы ранее; для сверки рангов печатаем также важность на `train`.

**Оговорки:** высокий коэффициент или большая важность **не доказывают причинно-следственную связь**. Кроме того, связанные между собой признаки (например, `temp_c` и `heating_season`) делят общий вклад друг с другом.

In [9]:
# Коэффициенты линейных моделей (с лагом и без него)
coef_lag = models.linear_coefficients(final_models["linreg"], FEATURES)
coef_no_lag = models.linear_coefficients(final_models["linreg_no_lag"], FEATURES_NO_LAG)
assert len(coef_lag) == len(FEATURES), "число коэффициентов linreg не совпало с FEATURES"
assert len(coef_no_lag) == len(FEATURES_NO_LAG), "число коэффициентов linreg_no_lag не совпало"

intercept_lag = float(final_models["linreg"].named_steps["linreg"].intercept_)
intercept_no_lag = float(final_models["linreg_no_lag"].named_steps["linreg"].intercept_)
rho_temp_heat = float(train["temp_c"].corr(train["heating_season"], method="spearman"))

print(f"Linear Regression (с pm25_lag1), intercept (средний прогноз на train) = {intercept_lag:.3f} µg/m³:")
print(coef_lag.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

print(f"\nLinear Regression (без pm25_lag1), intercept (средний прогноз на train) = {intercept_no_lag:.3f} µg/m³:")
print(coef_no_lag.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
print(f"\nКорреляция Спирмена temp_c и heating_season на train: rho = {rho_temp_heat:.3f}")

# Перестановочная важность Random Forest на test и для сверки рангов на train
importance_test = models.permutation_table(
    final_models["forest"], test[FEATURES], test[TARGET], N_REPEATS, SEED
)
importance_train = models.permutation_table(
    final_models["forest"], train[FEATURES], train[TARGET], N_REPEATS, SEED
)
assert len(importance_test) == len(FEATURES), "в importance_test должны быть все признаки"
assert len(importance_train) == len(FEATURES), "в importance_train должны быть все признаки"

imp_test_ranked = importance_test.assign(rank_test=range(1, len(importance_test) + 1))
imp_train_ranked = importance_train.assign(rank_train=range(1, len(importance_train) + 1))
importance_compare = imp_test_ranked.merge(
    imp_train_ranked,
    on="feature",
    suffixes=("_test", "_train"),
    validate="one_to_one",
)
assert len(importance_compare) == len(FEATURES), "при объединении потерялись признаки"
importance_compare = importance_compare[
    [
        "feature",
        "rank_test",
        "mae_increase_test",
        "mae_increase_std_test",
        "rank_train",
        "mae_increase_train",
        "mae_increase_std_train",
    ]
]

print("\nПерестановочная важность Random Forest (с pm25_lag1) — test и train рядом:")
print(importance_compare.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

importance_no_lag = models.permutation_table(
    final_models["forest_no_lag"], test[FEATURES_NO_LAG], test[TARGET], N_REPEATS, SEED
)
assert len(importance_no_lag) == len(FEATURES_NO_LAG), "в importance_no_lag должны быть все признаки"

print("\nПерестановочная важность Random Forest (без pm25_lag1) на test:")
print(importance_no_lag.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

# График 10: стандартизованные коэффициенты LR и перестановочная важность RF (с лагом)
palette = sns.color_palette()
fig, (ax_lr, ax_rf) = plt.subplots(1, 2, figsize=(18, 7))

lr_colors = [palette[0] if val >= 0 else palette[3] for val in coef_lag["coef_std"]]
ax_lr.barh(coef_lag["feature"], coef_lag["coef_std"], color=lr_colors)
ax_lr.axvline(0, color="gray", linestyle="--", linewidth=1.2)
ax_lr.invert_yaxis()
ax_lr.set_xlabel("Коэффициент, µg/m³ на 1 std признака")
ax_lr.set_title("Linear Regression (с лагом)")

ax_rf.barh(
    importance_test["feature"],
    importance_test["mae_increase"],
    xerr=importance_test["mae_increase_std"],
    color=palette[1],
    capsize=3,
)
ax_rf.axvline(0, color="gray", linestyle="--", linewidth=1.2)
ax_rf.invert_yaxis()
ax_rf.set_xlabel("Рост MAE при перемешивании, µg/m³")
ax_rf.set_title("Random Forest (с лагом, на test)")

top_lr_feat = coef_lag.loc[0, "feature"]
top_lr_val = float(coef_lag.loc[0, "coef_std"])
top_rf_feat = importance_test.loc[0, "feature"]
top_rf_val = float(importance_test.loc[0, "mae_increase"])
fig.suptitle(
    f"Обе модели сильнее всего опираются на {top_rf_feat} "
    f"(coef_std = {top_lr_val:.2f}, рост MAE = {top_rf_val:.2f} µg/m³)",
    y=1.02,
)
fig.tight_layout()
fig.savefig(f"{FIGURES_DIR}/10_importance.png", dpi=150, bbox_inches="tight")
plt.close(fig)

Linear Regression (с pm25_lag1), intercept (средний прогноз на train) = 31.016 µg/m³:
       feature  coef_std  coef_original
     pm25_lag1    19.278          0.596
        temp_c   -13.679         -1.268
        precip    -3.895         -1.007
      pressure    -2.049         -0.225
      humidity    -1.762         -0.096
heating_season    -1.620         -3.251
    wind_speed     0.946          2.135
    is_weekend    -0.133         -0.292

Linear Regression (без pm25_lag1), intercept (средний прогноз на train) = 31.016 µg/m³:
       feature  coef_std  coef_original
        temp_c   -30.650         -2.841
      humidity    -7.726         -0.422
      pressure    -5.292         -0.581
        precip    -3.772         -0.975
    wind_speed     0.552          1.245
    is_weekend    -0.331         -0.729
heating_season     0.183          0.368

Корреляция Спирмена temp_c и heating_season на train: rho = -0.795



Перестановочная важность Random Forest (с pm25_lag1) — test и train рядом:
       feature  rank_test  mae_increase_test  mae_increase_std_test  rank_train  mae_increase_train  mae_increase_std_train
     pm25_lag1          1              4.888                  0.330           1              10.886                   0.298
        temp_c          2              4.264                  0.340           2               7.594                   0.289
        precip          3              0.915                  0.158           3               2.407                   0.180
heating_season          4              0.508                  0.064           5               0.666                   0.036
      pressure          5              0.150                  0.047           7               0.589                   0.045
    wind_speed          6              0.118                  0.067           6               0.666                   0.061
    is_weekend          7              0.000            


Перестановочная важность Random Forest (без pm25_lag1) на test:
       feature  mae_increase  mae_increase_std
        temp_c        13.050             0.974
      humidity         1.585             0.248
heating_season         1.532             0.154
        precip         0.430             0.228
      pressure         0.294             0.073
    wind_speed         0.223             0.103
    is_weekend         0.016             0.016


**Что видим.** (вывод шага 8)

- **Топ-признаки `Linear Regression` с лагом (`intercept = 31.016 µg/m³`):** сильнее всего с прогнозом связаны вчерашний уровень `pm25_lag1` (`coef_std = +19.278` µg/m³ на 1 `std`, `coef_original = +0.596` µg/m³ на +1 µg/m³), температура `temp_c` (`coef_std = -13.679` µg/m³ на 1 `std`, `coef_original = -1.268` µg/m³ на +1 °C) и осадки `precip` (`coef_std = -3.895`, `coef_original = -1.007` µg/m³ на +1 мм). Далее идут `pressure` (`-2.049`, `-0.225` µg/m³/гПа), `humidity` (`-1.762`, `-0.096` µg/m³/%), `heating_season` (`-1.620`, `-3.251` µg/m³ при прочих равных), `wind_speed` (`+0.946`, `+2.135` µg/m³/м/с) и `is_weekend` (`-0.133`, `-0.292` µg/m³). Отрицательный коэффициент у `heating_season` объясняется сильной обратной связью с температурой (`ρ = -0.795` на `train`): оба признака описывают сезонность и делят вклад между собой.
- **Модель `Linear Regression` без лага (`intercept = 31.016 µg/m³`):** при удалении `pm25_lag1` главный вклад переходит к `temp_c` (`coef_std = -30.650`, `coef_original = -2.841` µg/m³ на +1 °C), `humidity` (`-7.726`, `-0.422` µg/m³/%), `pressure` (`-5.292`, `-0.581` µg/m³/гПа) и `precip` (`-3.772`, `-0.975` µg/m³/мм); коэффициент при `heating_season` становится `+0.183` (`+0.368` µg/m³), при `wind_speed` — `+0.552` (`+1.245` µg/m³/м/с), при `is_weekend` — `-0.331` (`-0.729` µg/m³).
- **Перестановочная важность `Random Forest` с лагом на `test`:** модель опирается прежде всего на `pm25_lag1` (рост `MAE` `+4.888 ± 0.330` µg/m³) и `temp_c` (`+4.264 ± 0.340` µg/m³), заметно слабее на `precip` (`+0.915 ± 0.158` µg/m³) и `heating_season` (`+0.508 ± 0.064` µg/m³). Вклад `pressure` (`+0.150 ± 0.047`), `wind_speed` (`+0.118 ± 0.067`), `is_weekend` (`0.000 ± 0.008`) и `humidity` (`-0.046 ± 0.078` µg/m³) близок к нулю.
- **Совпадение рангов на `train` и `test`:** первая тройка признаков совпадает полностью (`1` — `pm25_lag1`, `2` — `temp_c`, `3` — `precip`), но на `train` рост `MAE` примерно вдвое выше (`+10.886` и `+7.594` µg/m³ у двух лидеров), что согласуется с переобучением леса из шага 5. В нижней части списка ранги расходятся: например, `humidity` стоит на `4`-м месте на `train` (`+0.758` µg/m³), но опускается на `8`-е место на `test` (`-0.046` µg/m³) — разбиения по влажности, запомненные на обучении, не улучшают прогноз на новых сутках.
- **Перестановочная важность `Random Forest` без лага на `test`:** без вчерашнего PM2.5 главным ориентиром становится `temp_c` (`+13.050 ± 0.974` µg/m³), затем с большим отрывом идут `humidity` (`+1.585 ± 0.248` µg/m³) и `heating_season` (`+1.532 ± 0.154` µg/m³), а остальные признаки (`precip` `+0.430`, `pressure` `+0.294`, `wind_speed` `+0.223`, `is_weekend` `+0.016` µg/m³) дают менее `0.5` µg/m³.

![График 10](../figures/10_importance.png)

## Шаг 9. Остатки

Проверяем **остатки** моделей (L10) — разности `residual = факт − прогноз` (µg/m³). Нас интересуют три вопроса:
1. Нет ли систематического сдвига по времени (уходят ли ошибки в устойчивый плюс или минус в отдельные сезоны).
2. Меняются ли средний остаток и разброс ошибок при росте уровня прогноза (`Linear Regression`, сравнение по третям прогноза).
3. Независимы ли остатки соседних суток на лаге `1` (автокорреляция Спирмена `ρ`).

**Зачем накладывать остатки на полный календарь (`residuals_on_calendar`):** в тестовом периоде `18` из `501` календарных суток пропущены, а в `train` есть разрывы (включая `2022` год). Если считать сдвиг `shift(1)` прямо по таблице модели, сутки по разные стороны пропуска оказались бы «соседними». При `merge` с непрерывным календарём пропущенные дни получают `NaN`, и пары через дыру не учитываются.

In [10]:
# Календарные даты тестового периода и обучающей выборки
test_calendar = daily.loc[daily["date"] >= pd.Timestamp(TEST_START), "date"]
train_calendar = daily.loc[daily["date"] < pd.Timestamp(TEST_START), "date"]

# Автокорреляция остатков на лаге 1 для всех 6 прогнозов на test и для linreg на train
ac_rows = []
for name in MODEL_NAMES:
    res = test_predictions[TARGET] - test_predictions[name]
    frame = models.residuals_on_calendar(test_calendar, test_predictions["date"], res)
    assert len(frame) == len(test_calendar), "merge на календарь теста изменил число строк"
    ac = hypotheses.autocorrelation_by_lag(frame, "residual", [1])
    ac_rows.append(
        {
            "model": name,
            "rho_lag1": float(ac.loc[0, "rho"]),
            "n_pairs": int(ac.loc[0, "n"]),
        }
    )

res_lr_train = train[TARGET] - train_predictions["linreg"]
frame_lr_train = models.residuals_on_calendar(train_calendar, train["date"], res_lr_train)
assert len(frame_lr_train) == len(train_calendar), "merge на календарь train изменил число строк"
ac_lr_train = hypotheses.autocorrelation_by_lag(frame_lr_train, "residual", [1])
ac_rows.append(
    {
        "model": "linreg (train)",
        "rho_lag1": float(ac_lr_train.loc[0, "rho"]),
        "n_pairs": int(ac_lr_train.loc[0, "n"]),
    }
)

residual_autocorr = pd.DataFrame(ac_rows)
assert len(residual_autocorr) == 7, "в таблице автокорреляции должно быть 7 строк"
assert (residual_autocorr["n_pairs"] > 0).all(), "число пар для лага 1 должно быть > 0"

print(
    f"Проверка merge по календарю пройдена: test = {len(test_calendar)} суток, "
    f"train = {len(train_calendar)} суток."
)
print("\nАвтокорреляция остатков на лаге 1 (Спирмен по календарю):")
print(residual_autocorr.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

# Средний остаток и MAE по третям прогноза Linear Regression на тесте
res_lr = test_predictions[TARGET] - test_predictions["linreg"]
res_rf = test_predictions[TARGET] - test_predictions["forest"]

thirds_frame = pd.DataFrame(
    {
        "pred": test_predictions["linreg"],
        "residual": res_lr,
        "abs_err": res_lr.abs(),
    }
)
thirds_frame["third"] = pd.qcut(thirds_frame["pred"], 3)
lr_thirds = (
    thirds_frame.groupby("third", observed=False)
    .agg(
        n=("residual", "size"),
        mean_residual=("residual", "mean"),
        mae=("abs_err", "mean"),
        std_residual=("residual", "std"),
    )
    .reset_index()
)
assert int(lr_thirds["n"].sum()) == len(test_predictions), "сумма по третям != len(test)"

print("\nОстатки Linear Regression на тесте по третям прогноза:")
print(lr_thirds.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

# График 17: остатки по датам теста и остатки linreg против прогноза
rho_lr_test = float(
    residual_autocorr.loc[residual_autocorr["model"] == "linreg", "rho_lag1"].iloc[0]
)
rho_rf_test = float(
    residual_autocorr.loc[residual_autocorr["model"] == "forest", "rho_lag1"].iloc[0]
)
mae_low_third = float(lr_thirds.loc[0, "mae"])
mae_high_third = float(lr_thirds.loc[2, "mae"])

palette = sns.color_palette()
fig, (ax_time, ax_pred) = plt.subplots(1, 2, figsize=(18, 6))

ax_time.plot(
    test_predictions["date"],
    res_lr,
    color=palette[0],
    linewidth=1.2,
    alpha=0.8,
    label="Linear Regression",
)
ax_time.plot(
    test_predictions["date"],
    res_rf,
    color=palette[1],
    linewidth=1.2,
    alpha=0.8,
    label="Random Forest",
)
ax_time.axhline(0, color="gray", linestyle="--", linewidth=1.2)
ax_time.tick_params(axis="x", rotation=20)
ax_time.set_xlabel("Дата")
ax_time.set_ylabel("Остаток (факт − прогноз), µg/m³")
ax_time.set_title("Остатки по датам теста")
ax_time.legend(loc="upper right")

ax_pred.scatter(test_predictions["linreg"], res_lr, alpha=0.55, s=28, color=palette[0])
ax_pred.axhline(0, color="gray", linestyle="--", linewidth=1.2)
ax_pred.set_xlabel("Прогноз, µg/m³")
ax_pred.set_ylabel("Остаток, µg/m³")
ax_pred.set_title("Остатки Linear Regression против прогноза")

fig.suptitle(
    f"При высоком прогнозе разброс растёт (MAE {mae_low_third:.1f} → {mae_high_third:.1f} µg/m³); "
    f"автокорреляция остатков LR ρ = {rho_lr_test:.3f}, RF ρ = {rho_rf_test:.3f}",
    y=1.02,
)
fig.tight_layout()
fig.savefig(f"{FIGURES_DIR}/17_residuals.png", dpi=150, bbox_inches="tight")
plt.close(fig)

Проверка merge по календарю пройдена: test = 501 суток, train = 1543 суток.

Автокорреляция остатков на лаге 1 (Спирмен по календарю):
         model  rho_lag1  n_pairs
         naive    -0.141      480
    month_mean     0.527      480
        linreg     0.174      480
        forest     0.088      480
 linreg_no_lag     0.655      480
 forest_no_lag     0.563      480
linreg (train)     0.046      840

Остатки Linear Regression на тесте по третям прогноза:
            third   n  mean_residual    mae  std_residual
(-11.038, 10.432] 161          5.296  5.479         4.450
 (10.432, 25.847] 161         -1.081  5.297         6.727
(25.847, 107.919] 161         -3.097 15.988        20.256


**Что видим.** (вывод шага 9)

- **Автокорреляция остатков на лаге `1` (Спирмен по календарю):**
  - на тесте (`n_pairs = 480`): у моделей с лагом остаётся слабая положительная связь соседних суток — `linreg` (`ρ = 0.174`), `forest` (`ρ = 0.088`), тогда как у `naive` корреляция отрицательная (`ρ = -0.141`);
  - у моделей без вчерашнего PM2.5 автокорреляция остатков на тесте высокая: `linreg_no_lag` (`ρ = 0.655`), `forest_no_lag` (`ρ = 0.563`), `month_mean` (`ρ = 0.527`) — без `pm25_lag1` модели не учитывают суточную инерцию загрязнения;
  - на `train` у `linreg` (`n_pairs = 840`) связь на лаге `1` составляет `ρ = 0.046`.
- **Остатки по времени и по третям прогноза `Linear Regression` (по `n = 161` суток в каждой трети):**
  - нижняя треть `(-11.038, 10.432]` µg/m³: средний остаток `+5.296` µg/m³, `MAE = 5.479` µg/m³, `std = 4.450` µg/m³ (летом линейная модель систематически занижает прогноз и местами уходит ниже нуля, поэтому `факт − прогноз > 0`, что видно и на левой панели);
  - средняя треть `(10.432, 25.847]` µg/m³: средний остаток `-1.081` µg/m³, `MAE = 5.297` µg/m³, `std = 6.727` µg/m³;
  - верхняя треть `(25.847, 107.919]` µg/m³: средний остаток `-3.097` µg/m³, `MAE = 15.988` µg/m³, `std = 20.256` µg/m³ — при высоком прогнозе (зимой) разброс ошибок и `MAE` вырастают почти втрое (на графике справа видно характерное «веерное» расширение).
- **Что это означает для выводов:** остатки соседних суток не являются полностью независимыми (особенно при отсутствии лага и у `linreg` на тесте с `ρ = 0.174`). Поэтому доверительные интервалы парного bootstrap в шаге 6 несколько узковаты, а стандартные `p`-значения коэффициентов линейной регрессии мы не приводим.

![График 17](../figures/17_residuals.png)

## Итог

1. **Лучшая модель на тесте и сравнение с наивным прогнозом:** на тестовом периоде (`n = 483` суток, `2024-07-01 … 2025-11-13`) минимальную среднюю абсолютную ошибку показал `Random Forest` с лагом `pm25_lag1`: `MAE = 8.097` µg/m³ (`RMSE = 13.653` µg/m³, `R² = 0.644`) против `MAE = 8.592` µg/m³ (`RMSE = 14.828`, `R² = 0.581`) у наивного прогноза `naive` и `MAE = 8.921` µg/m³ (`RMSE = 13.054`, `R² = 0.675`) у `Linear Regression` с лагом (вывод шага 5). По парному bootstrap (`5000` выборок) точечная разность `MAE` для `forest` против `naive` равна `-0.495` µg/m³, 95 % доверительный интервал `[-1.326, 0.313]` µg/m³ (доля `< 0` — `0.887`), для `linreg` против `naive` — `+0.330` µg/m³, 95 % CI `[-0.270, 0.910]` µg/m³ (вывод шага 6). Оба интервала включают ноль, поэтому на этих данных не доказано, что модель точнее правила «завтра как сегодня».
2. **Вклад только погоды и календаря (модели без лага против `month_mean` и `naive`):** в моделях без вчерашнего уровня `pm25_lag1` ошибки заметно выше: у `forest_no_lag` на тесте `MAE = 10.765` µg/m³ (`R² = 0.354`), у `linreg_no_lag` — `MAE = 12.830` µg/m³ (`R² = 0.453`), тогда как простое среднее по месяцу обучения `month_mean` даёт `MAE = 9.969` µg/m³ (`R² = 0.454`) (вывод шага 5). Обе модели без лага заметно уступают наивному прогнозу (`+2.173`, 95 % CI `[0.887, 3.469]` у `forest_no_lag` и `+4.238`, 95 % CI `[3.035, 5.364]` у `linreg_no_lag`); `linreg_no_lag` хуже и `month_mean` (`+2.861`, 95 % CI `[1.944, 3.742]`), а для `forest_no_lag` против `month_mean` разность `+0.796` µg/m³ с 95 % CI `[-0.454, 2.068]` включает ноль (вывод шага 6). Одна фактическая погода с календарём без вчерашнего PM2.5 не превосходит даже среднее месяца.
3. **Согласованность скользящей проверки на `train` и оценки на `test`:** порядок моделей по `MAE` совпадает — и в 5-фолдовом CV на `train` (`n = 931`), и на `test` с лагом лидирует `forest`, далее близко идут `naive` и `linreg`, а варианты без лага заметно отстают (выводы шагов 3–4 и 5). При этом на тесте ошибки в среднем ниже, чем в CV на `train`: в CV `MAE` составлял `9.666` у `forest`, `11.314` у `linreg`, `11.562` у `naive`, `12.326` у `forest_no_lag` и `16.841` у `linreg_no_lag` (вывод шагов 3–4), на тесте — `8.097`, `8.921`, `8.592`, `10.765` и `12.830` µg/m³ соответственно (вывод шага 5). В CV `forest` опережал `naive` в `5` из `5` фолдов, а `linreg` — в `4` из `5`, тогда как на одном тестовом отрезке `linreg` по `MAE` чуть уступил наивному (`8.921` против `8.592`), что согласуется с сильным разбросом оценок между отдельными временными периодами (`8.240–14.471` µg/m³ у `naive` по фолдам, вывод шагов 3–4).
4. **На что опираются модели:** обе модели с лагом сильнее всего опираются на вчерашний уровень загрязнения `pm25_lag1` и температуру `temp_c`, а на третьем месте идут осадки `precip` (вывод шага 8). У `Linear Regression` стандартизованные коэффициенты `coef_std` равны `+19.278` µg/m³ для `pm25_lag1` (`coef_original = +0.596`), `-13.679` µg/m³ для `temp_c` (`-1.268` µg/m³ на +1 °C) и `-3.895` µg/m³ для `precip` (`-1.007` µg/m³ на +1 мм); в перестановочной важности `Random Forest` на тесте перемешивание `pm25_lag1` соответствует росту `MAE` на `+4.888 ± 0.330` µg/m³, `temp_c` — на `+4.264 ± 0.340` µg/m³, `precip` — на `+0.915 ± 0.158` µg/m³ (вывод шага 8). Без лага главным ориентиром становится `temp_c` (`coef_std = -30.650` у `linreg_no_lag` и рост `MAE` `+13.050 ± 0.974` µg/m³ у `forest_no_lag`, вывод шага 8).
5. **Проверка остатков:** у моделей без лага остатки сильно автокоррелированы на лаге `1` (`ρ = 0.655` у `linreg_no_lag`, `ρ = 0.563` у `forest_no_lag`, `ρ = 0.527` у `month_mean`), тогда как в моделях с `pm25_lag1` эта связь существенно ниже: `ρ = 0.174` у `linreg` и `ρ = 0.088` у `forest` на тесте (`n_pairs = 480`; на `train` у `linreg` `ρ = 0.046`, вывод шага 9). В верхней трети прогнозов разброс и величина ошибок заметно выше, чем в нижней трети: у `linreg` от нижней трети (`MAE = 5.479` µg/m³, средний остаток `+5.296` — занижение летом) к верхней трети (`MAE = 15.988` µg/m³, `std = 20.256` µg/m³ — крупные ошибки зимой), что согласуется с более высоким `MAE` в отопительный сезон (`13.725` против `5.132` µg/m³ у `linreg` и `14.583` против `2.981` µg/m³ у `forest`, выводы шагов 5 и 9).

**Ограничения:**
- **Одна станция и один тестовый период:** проверка выполнена по данным одной станции на отрезке `2024-07-01 … 2025-11-13` (`483` суток), куда входит только один полный отопительный сезон (`213` отопительных суток, вывод шага 2).
- **Зависимость соседних суток:** остатки моделей на тесте сохраняют автокорреляцию на лаге `1` (`ρ = 0.174` у `linreg` и `ρ = 0.088` у `forest`, вывод шага 9), поэтому доверительные интервалы парного bootstrap в шаге 6 несколько узковаты, а стандартные `p`-значения коэффициентов не считались.
- **Фактическая погода дня `D`:** модели используют измеренные суточные метеоданные целевого дня, а не реальный метеопрогноз, поэтому полученные метрики — оценка сверху для настоящего прогноза на завтра.
- **Потеря суток из-за пропусков и лага:** из `2 044` суток в `daily.csv` выпадают `510` суток без `pm25_mean` и ещё `120` суток с измерением, но без вчерашнего значения `pm25_lag1` (всего потеряно `630` строк, в том числе `18` из `501` календарных суток тестового периода, выводы шагов 1 и 2).
- **Важность и коэффициенты ≠ причина:** высокие значения коэффициентов `Linear Regression` и перестановочной важности `Random Forest` показывают, на какие признаки опирается модель, но не доказывают причинно-следственную связь.
- **Связанные признаки делят вклад:** температура `temp_c` и флаг `heating_season` сильно коррелируют (`ρ = -0.795` на `train`, вывод шага 8), поэтому их индивидуальные коэффициенты и оценки важности распределяют общий сезонный вклад между собой.
- **Переобучение `Random Forest`:** на обучающей выборке ошибка `forest` ниже, чем на тесте (`mae_train = 6.552` против `8.097` µg/m³ на `test`, у `forest_no_lag` — `8.201` против `10.765` µg/m³), а оценки важности на `train` примерно вдвое выше тестовых (`+10.886` против `+4.888` µg/m³ для `pm25_lag1`, выводы шагов 5 и 8).
